In [ ]:
from resources.imports import *
from pathlib import Path
from openpyxl import load_workbook

from resources.lattices import Geometry, effProperties
from resources.calculations import get_ductileData, get_fractureData, calcUT, calcFT, plot_curve, stress_to_mpa


In [ ]:
LAT = "tri"
MODEL = "both"
l = 10.0     # mm
t = None     # mm   # t is optional and should also be supplied in mm when used.
rD = 0.2
material = "Ti"
dis = 'per'
dn = 0.2
nSim = '1'

nnx = 30 if LAT.lower() == "tri" else 20
nnx = nnx

if material.lower() == "ti":
    E_s = 123e9  # Pa; effProperties/calcC_mohr expect the base-material modulus in Pa.
    v_s = 0.3
    s_ts_mpa, s_ys_mpa = 1111.792033, 932.0  # MPa
elif material.lower() == "sic":
    E_s = 410e9  # Pa
    v_s = 0.14
    s_ts_mpa, s_ys_mpa = 550.0, 550.0  # MPa

if dis == "per":
    dn = ""

val = False
psc = False
mesh = False

data = False
PATH_ADD = "sApp"

root_data = Path("Z:/p1/data") / material
root_sims = Path("Z:/p1/sims") / material

if data:
    os.chdir(root_data / dis / PATH_ADD / str(dn) / LAT)
else:
    os.chdir(root_sims / PATH_ADD)
    # os.chdir(r"C:/temp")

if val:
    os.chdir(root_sims / "sApp")
    dis, nSim = 'per', '1'
elif psc:
    os.chdir(root_sims / "PSC" / str(int(l)))
    dis, nSim = 'per', '1'
elif mesh:
    brackets = 1
    coarse = 1
    fine = 1
    os.chdir(root_sims / "MeshConv" / f"{brackets}-{coarse}-{fine}")
    dis, nSim = 'per', '1'


In [ ]:
# ## CONVERGENCE STUDIES

# sim = 20

# start_row = 94
# cell_row = start_row + sim

# sheet_name = "PSC"
# column = ["G", "H", "I", "K", "L", "M", "N"]

# nnxs = [10, 16, 20, 26, 30, 36, 40]
# nnx = nnxs[sim]


# sheet_name = "MeshConv"
# column = ["K", "L", "M", "O", "P", "Q", "R"]

# coarses = [1, 2, 5]
# fines = [1, 2, 5, 10 ,15, 20, 25]

# coarse = coarses[sim//len(fines)]
# fine = fines[sim%len(fines)]

# os.chdir(r"Z:\\p1\\sims\\Ti\MeshConv\\1-"+str(coarse)+"-"+str(fine))

In [ ]:
geom = Geometry(LAT, l, nnx, rD=rD, t=t)
geom.FTcalc()

rD = geom.rD
C = None
E_eff, v_eff, E_eff_pe, v_eff_pe = effProperties(LAT, geom, E_s=E_s, v_s=v_s)

# Optional legacy closed-form stiffness overrides for square variants.
# The default current workflow lets effProperties()/calcFT() derive stiffness from Geometry.
if LAT.lower() == "45square" and t is not None:
    S = (1 / E_s) * np.array([
        [0.5 * (l / t) + 0.5 * (l / t) ** 3, 0.5 * (l / t) - 0.5 * (l / t) ** 3 + 1e-7, 0],
        [0.5 * (l / t) - 0.5 * (l / t) ** 3, 0.5 * (l / t) + 0.5 * (l / t) ** 3, 0],
        [0, 0, 2 * (l / t)],
    ])
    C = np.linalg.inv(S)
elif LAT.lower() == "square":
    S = np.array([
        [1 / E_eff, -v_eff / E_eff, 0],
        [-v_eff / E_eff, 1 / E_eff, 0],
        [0, 0, 1 / (E_s * (1 / 16) * (rD ** 3))],
    ])
    C = np.linalg.inv(S)

if C is not None:
    E_eff, v_eff, E_eff_pe, v_eff_pe = effProperties(LAT, geom, E_s=E_s, v_s=v_s, C=C)

print("Geometry:")
print(f"  LAT={LAT}, nnx={geom.nnx}, nny={geom.nny}, l={geom.l:g} mm, rD={geom.rD:g}, iso={geom.iso}")
print("Effective properties:")
print(f"  E_eff    = {E_eff:.6g} Pa ({stress_to_mpa(E_eff):.6g} MPa)")
print(f"  v_eff    = {v_eff:.6g}")
print(f"  E_eff_pe = {E_eff_pe:.6g} Pa ({stress_to_mpa(E_eff_pe):.6g} MPa)")
print(f"  v_eff_pe = {v_eff_pe:.6g}")


In [ ]:
os.getcwd(), rD, geom.iso


In [ ]:
if MODEL.lower() == "ductile" or MODEL.lower() == "both": 
    CSVout = Path("transfer") / f"OUT-Ductile-{LAT}-{int(nnx)}-{dis}-{nSim}.csv"
    UTdf = get_ductileData(CSVout, crit=0.25)

if MODEL.lower() == "fracture" or MODEL.lower() == "both":
    CSVout = Path("transfer") / f"OUT-Fracture-{LAT}-{int(nnx)}-{dis}-{nSim}.csv"
    FTdf = get_fractureData(CSVout)


In [ ]:
if MODEL.lower() == "ductile" or MODEL.lower() == "both":
    ductility, strength, stiffness, WoF = calcUT(UTdf)
    print("DUCT:")
    print(f"  Ductility [-]: {ductility:.6g}")
    print(f"  Strength [MPa]: {strength:.6g}")
    print(f"  Stiffness [MPa]: {stiffness:.6g}")
    print(f"  WoF [MPa]: {WoF:.6g}")
    print()

if MODEL.lower() == "fracture" or MODEL.lower() == "both":    
    P, dd, Ks, Kjs = calcFT(FTdf, geom, E_eff_pe, n_Ks=1, validation=val, E=E_s, C=C)
    Kic, Kjic = Ks[0], Kjs[0]
    Kic_norm_l = Kic / (s_ts_mpa * np.sqrt(l))
    Kjic_norm_l = Kjic / (s_ts_mpa * np.sqrt(l))
    print("FRAC:")
    print(f"  Force [N]: {P:.6g}")
    print(f"  Displacement [mm]: {dd:.6g}")
    print(f"  K_IC [MPa*sqrt(mm)]: {Kic:.6g}")
    print(f"  K_JIC [MPa*sqrt(mm)]: {Kjic:.6g}")
    print(f"  K_IC / (sigma_ts*sqrt(l)): {Kic_norm_l:.6g}")
    print(f"  K_JIC / (sigma_ts*sqrt(l)): {Kjic_norm_l:.6g}")


In [ ]:
if MODEL.lower() == "ductile" or MODEL.lower() == "both":
    plot_curve([UTdf], typ="ut", label="Ductile")

if MODEL.lower() == "fracture" or MODEL.lower() == "both":
    plot_curve([FTdf], typ="ft", label="Fracture")

In [ ]:
# ### WRITE TO EXCEL

# file_path = "C:/Users/exy053/OneDrive - Queen Mary, University of London/Documents/Research/p1-LatticeFractureToughness/Ductility-FractureToughness.xlsx"
# cell_coords = [f"{column[0]}{cell_row}", f"{column[1]}{cell_row}", f"{column[2]}{cell_row}", f"{column[3]}{cell_row}", f"{column[4]}{cell_row}", f"{column[5]}{cell_row}", f"{column[6]}{cell_row}"]
# values_to_write = [stiffness, strength, ductility, P, dd, Kic, Kjic]

# # --- Load workbook and target sheet ---
# wb = load_workbook(file_path)
# ws = wb[sheet_name]

# # --- Write values ---
# for coord, value in zip(cell_coords, values_to_write):
#     ws[coord] = value
    
# # --- Save workbook ---
# wb.save(file_path)
# wb.close()


In [ ]:
# ### SIM COMPARISON

# append = True
# appendReset = False
# if appendReset:
#     UTsave, FTsave = [], []

# if MODEL.lower() == "ductile" or MODEL.lower() == "both":
#     if append:
#         UTsave.append([UTdf, ductility, strength, stiffness])
#         plot_curve(np.array(UTsave, dtype=object)[:,0], typ="UT")


# if MODEL.lower() == "fracture" or MODEL.lower() == "both":
#     if append:
#         FTsave.append([FTdf, P, dd, Ks, Kjs])
#         plot_curve(np.array(FTsave, dtype=object)[:,0], typ="FT")

In [ ]:
# ### NORMALIZATION of Ks

# s_ts_mpa = 932 * (1 + (0.1105 - (932 / 123000)) ** 0.7237)  # MPa

# K_norm = Kic / s_ts_mpa
# K_norml = Kic / (s_ts_mpa * np.sqrt(l))
# Kj_norm = Kjic / s_ts_mpa
# Kj_norml = Kjic / (s_ts_mpa * np.sqrt(l))

# K_norm, K_norml, Kj_norm, Kj_norml
